<a href="https://colab.research.google.com/github/wingated/cs473/blob/main/cs473/labs/cs473_new_lda_lab_week_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/wingated/cs473/blob/main/labs/cs473_lab_week_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a><p><b>After clicking the "Open in Colab" link, copy the notebook to your own Google Drive before getting started, or it will not save your work</b></p>

# BYU CS 473 Lab Week 4

## Latent Dirichlet Allocation (LDA)
The Anti-Online Reading Society has struck Wikipedia! Their target: every article title.
Little did they know, you have a secret weapon: LDA. With it, you can organize every article into topics, clustering them automatically and making sense of the chaos.


 ---
 ## Exercise #1: Create a   gibbs sampler
 ### Gibbs sampling on a 6-sided die:
 Given a probability distribution (p1, …, pn), this random number generator, denoted Random(p1, …, pn), models an n-sided biased die and returns integer i with probability pi. For example, the standard six-sided fair die represents the random number generator Random(1/6, 1/6, 1/6, 1/6, 1/6, 1/6), whereas a biased die might represent the random number generator Random(0.1, 0.2, 0.3, 0.05, 0.1, 0.25). GibbsSampler() further generalizes the random number generator by using the function Random(p1, …, pn) defined for any set of non-negative numbers, i.e., not necessarily satisfying the condition that the pi sum to 1. If the pi sum to some C > 0 instead, then Random(p1, …, pn) is defined as Random(p1/C, …, pn/C), where (p1/C, …, pn/C) is a probability distribution. For example, for (0.1, 0.2, 0.3) with 0.1 + 0.2 + 0.3 = 0.6.

 Now code up your own Gibbs sampler using the following scaffold:

In [2]:
import random


def gibbs_sampler(data, k, t, n):
    """
    Scaffold for a Gibbs sampler.

    Arguments:
    - data: list of sequences (or generic observations)
    - k: length of motif / chunk / window (if applicable)
    - t: number of sequences (or items)
    - n: number of iterations

    Returns:
    - BestMotifs: the "best" sampled patterns
    """

    # Initialize motifs randomly
    Motifs = []
    for seq in data:
        start = random.randint(0, len(seq) - k)
        Motifs.append(seq[start:start+k])
    BestMotifs = Motifs.copy()

    # Gibbs sampling iterations
    for iteration in range(n):
        # Randomly choose one sequence to resample
        i = random.randint(0, t - 1)

        # Construct profile / conditional probabilities
        # Pseudocode:
        # Profile = generate_profile(Motifs excluding i-th sequence)


        # Sample new motif / chunk for i-th sequence

        # Motifs[i] = sample_from_profile(sequence[i], Profile)

        # Step 2d: Update best motifs if score improved
        # Pseudocode:
        #   if Score(Motifs) < Score(BestMotifs):
        #       BestMotifs = Motifs


    # Step 3: Return the best motifs found
    return BestMotifs


# optional helper function scafolds
def generate_profile(motifs):
    """
    Pseudocode:
    - Create a profile (conditional probability distribution)
      from the current set of motifs.
    - Usually includes pseudocounts to avoid zeros.
    """
    profile = []
    # TODO: implement profile calculation
    return profile


def profile_random_kmer(sequence, k, profile):
    """
    Pseudocode:
    - Enumerate all k-length chunks in sequence
    - Compute probability for each chunk using the profile
    - Normalize probabilities
    - Randomly select a chunk according to probabilities
    """
    # TODO: implement weighted random sampling
    return sequence[0:k]  # placeholder


def Score(motifs):
    """
    Pseudocode:
    - Compute a score for a set of motifs
      (e.g., distance from consensus)
    """
    # TODO: implement scoring
    return 0  # placeholder


### Test your sampler with the following code. A correctly coded sampler should converge to a random dice value.
### For example:


	1: 0
	2: 0
	3: 11
	4: 5
	5: 0
	6: 34

In [6]:
def gibbs_dice(dice_count=50, faces=6, iterations=2000):
    """
    Test the Gibbs sampler using dice sequences.
    Each die is a sequence of its faces: "123456"
    """

    dice_sequences = [''.join(str(f) for f in range(1, faces+1)) for _ in range(dice_count)]
    t = len(dice_sequences)
    k = 1

    best_faces = gibbs_sampler(dice_sequences, k=k, t=t, n=iterations)


    return [int(face) for face in best_faces]


dice = gibbs_dice()
faces = [1,2,3,4,5,6]
print(f"Gibbs dice sampled faces: {dice}\nCounts:")
for face in faces:
    print(f"\t{face}: {dice.count(face)}")


Gibbs dice sampled faces: [1, 4, 4, 3, 1, 3, 3, 5, 2, 5, 5, 5, 3, 1, 2, 4, 1, 4, 5, 4, 1, 3, 5, 5, 6, 2, 3, 4, 2, 6, 4, 1, 3, 1, 3, 2, 5, 1, 2, 1, 4, 5, 5, 5, 6, 1, 5, 2, 5, 5]
Counts:
	1: 10
	2: 7
	3: 8
	4: 8
	5: 14
	6: 3


## Discussion:
Why is convergence important for a gibbs sampler?

Your response here

---
## Exercise #2: LDA relation to "weighted dice"

| Weighted Dice                                          | LDA                                                                      |
| ------------------------------------------------- | ------------------------------------------------------------------------ |
| Dice → sequences                                  | Documents                                                                |
| Faces → possible outcomes                         | Topics                                                                   |
| Motifs (current face) → current assignment        | Topic assignment for each word                                           |
| Profile → conditional probability from other dice | Conditional probability of assigning a word to a topic (based on counts) |
| Iterations → repeated resampling                  | Iteratively resample topic assignments                                   |
| Convergence → clusters of dice                    | Convergence → stable topic assignments per word                          |

## Use your gibbs sampler code to apply to an LDA below:
Remember: words should converge to specific topics.

In [12]:
from collections import defaultdict

def lda_gibbs(documents, num_topics=5, iterations=2000):
    """
    Create an LDA Gibbs Sampler.
    documents: list of lists of words (documents)
    num_topics: number of topics
    """
    # Initialization
    topic_assignments = []  # list of lists, same shape as documents
    doc_topic_counts = [defaultdict(int) for _ in documents]  # counts of topics per doc
    topic_word_counts = [defaultdict(int) for _ in range(num_topics)]  # counts of words per topic
    topic_counts = [0]*num_topics  # total words assigned to each topic


    return topic_assignments, doc_topic_counts, topic_word_counts

In [13]:
# Example documents with 5 words each
docs = [
    ["apple", "banana", "apple", "banana", "cherry"],
    ["banana", "apple", "banana", "banana", "cherry"],
    ["apple", "apple", "banana", "apple", "cherry"],
    ["banana", "cherry", "banana", "apple", "cherry"],
    ["apple", "cherry", "cherry", "banana", "apple"]
]

# Run Gibbs sampler for LDA
topic_assignments, doc_topic_counts, topic_word_counts = lda_gibbs(docs,num_topics=2,iterations=2000)

print("Topic assignments per document:")
for i, topics in enumerate(topic_assignments):
    print(f"Doc {i+1}: {topics}")

print("\nDocument-topic counts:")
for i, counts in enumerate(doc_topic_counts):
    print(f"Doc {i+1}: {dict(counts)}")

print("\nTopic-word counts:")
for k, counts in enumerate(topic_word_counts):
    print(f"Topic {k}: {dict(counts)}")

Topic assignments per document:

Document-topic counts:
Doc 1: {}
Doc 2: {}
Doc 3: {}
Doc 4: {}
Doc 5: {}

Topic-word counts:
Topic 0: {}
Topic 1: {}



## Discussion:
What is an LDA? Why do we use a Gibbs Sampler?

Your response here

---
## Exercise #3: create your own documents

Create your own set of at least 3 documents with at least 5 words in each. Do this with 3 topics.

In [ ]:
# Your code here

## Discussion
What difference did choosing 3 topics make? Name at least one real application that an LDA can be used for.

Your response here